# CLIP Residual Fusion Hyperparameter Tuning

In [1]:
import sys
from pathlib import Path

IN_COLAB = 'google.colab' in sys.modules

if IN_COLAB:
    from google.colab import drive
    drive.mount('/content/drive')
    PROJECT_ROOT = Path('/content/drive/MyDrive/dissertation')
else:
    PROJECT_ROOT = Path('../../..').resolve()

sys.path.insert(0, str(PROJECT_ROOT))

print(f'IN_COLAB={IN_COLAB}')
print(f'PROJECT_ROOT={PROJECT_ROOT}')

Mounted at /content/drive
IN_COLAB=True
PROJECT_ROOT=/content/drive/MyDrive/dissertation


In [2]:
import hashlib
import json
import time

import numpy as np
import pandas as pd
import torch

from src import dataloader as data
from src import metrics
from src import train
from src.models.classical_baseline import HORIZONS
from src.models.multimodal_residual import load_frozen_baseline, FrozenImageOnlyResidualPredictor

ENCODER_NAME = 'clip'
IMG_FEAT_DIM = 768
EXPECTED_IMG_PROJ_DIM = 4

SEEDS = [7, 14, 21, 28, 35, 42, 49, 56, 63, 70]

EXP_DIR = PROJECT_ROOT / 'experiments' / '5_residual_hyper'

SHARED_CONFIG_PATH = EXP_DIR / 'shared_search' / 'multimodal_tuning_configurations.csv'

RESULTS_DIR = EXP_DIR / 'results' / ENCODER_NAME
CHECKPOINT_ROOT = EXP_DIR / 'checkpoints' / ENCODER_NAME

BASELINE_CHECKPOINT_DIR = PROJECT_ROOT / 'experiments' / '3_initial_multimodal' / 'multimodal' / 'checkpoints' / 'gated_cgm_time'

BASELINE_RESULTS_DIR = PROJECT_ROOT / 'experiments' / '3_initial_multimodal' / 'multimodal' / 'results' / 'gated_cgm_time'

SEED_RESULT_DIR = RESULTS_DIR / 'seed_results'
SUMMARY_DIR = RESULTS_DIR / 'summary'

for p in [RESULTS_DIR, SEED_RESULT_DIR, SUMMARY_DIR, CHECKPOINT_ROOT]:
    p.mkdir(parents=True, exist_ok=True)

TRIAL_TABLE_PATH = SUMMARY_DIR / 'clip_tuning_trials.csv'
BEST_CONFIG_PATH = SUMMARY_DIR / 'clip_best_config.json'
PARTICIPANT_TABLE_PATH = SUMMARY_DIR / 'clip_selected_config_participant_rmse.csv'

print('Shared configs:', SHARED_CONFIG_PATH)
print('Results:', RESULTS_DIR)

Shared configs: /content/drive/MyDrive/dissertation/experiments/5_residual_hyper/shared_search/multimodal_tuning_configurations.csv
Results: /content/drive/MyDrive/dissertation/experiments/5_residual_hyper/results/clip


In [6]:
configs = pd.read_csv(SHARED_CONFIG_PATH).sort_values('config_order').reset_index(drop=True)

required_cols = {
    'config_order', 'config_id', 'is_anchor',
    'img_proj_dim', 'img_dropout', 'lr', 'weight_decay'
}

expected_anchor = {
    'img_proj_dim': 4,
    'img_dropout': 0.3,
    'lr': 1e-4,
    'weight_decay': 1e-5,
}

anchor_rows = configs[configs['is_anchor'].astype(str).str.lower().isin(['true','1'])]

config_sha256 = hashlib.sha256(SHARED_CONFIG_PATH.read_bytes()).hexdigest()

print(f'Loaded {len(configs)} fixed configurations.')
print('CSV SHA256:', config_sha256)
display(configs)

Loaded 40 fixed configurations.
CSV SHA256: 32e220271d6157119aec540c96162224e751d9f49207f511dc6cb171c77434d3


,config_order,config_id,is_anchor,img_proj_dim,img_dropout,lr,weight_decay
0,1,cfg_01,True,4,0.3,0.00010,0.000010
1,2,cfg_02,False,4,0.1,0.00003,0.000000
2,3,cfg_03,False,4,0.5,0.00003,0.000010
3,4,cfg_04,False,4,0.1,0.00001,0.000001
4,5,cfg_05,False,4,0.3,0.00100,0.000000
5,6,cfg_06,False,4,0.5,0.00100,0.000001
6,7,cfg_07,False,4,0.5,0.00010,0.000001
7,8,cfg_08,False,4,0.1,0.00100,0.000001
8,9,cfg_09,False,4,0.0,0.00010,0.000100
9,10,cfg_10,False,4,0.5,0.00003,0.000100


In [7]:
stats = data.load_cgm_stats()

train_loader = data.load_multimodal_tensors('train', encoder_name=ENCODER_NAME)
val_loader = data.load_multimodal_tensors('val', encoder_name=ENCODER_NAME, shuffle=False)

val_participant_ids = data.load_multimodal_participant_ids('val')
VAL_PARTICIPANT_IDS = sorted(set(val_participant_ids))

print('n_train_batches:', len(train_loader))
print('n_val_batches:', len(val_loader))
print('n_val_samples:', len(val_loader.dataset))
print('validation participants:', VAL_PARTICIPANT_IDS)

sample_batch = next(iter(train_loader))
print('batch shapes:', [tuple(x.shape) for x in sample_batch])

n_train_batches: 323
n_val_batches: 64
n_val_samples: 16355
validation participants: [np.str_('CGMacros-005'), np.str_('CGMacros-016'), np.str_('CGMacros-027'), np.str_('CGMacros-044'), np.str_('CGMacros-046'), np.str_('CGMacros-048')]
batch shapes: [(256, 24), (256,), (256,), (256, 768), (256, 5)]


In [8]:
def forward_fn(model, batch, device):
    cgm, tsm, has_meal, img_feat, target = batch
    pred = model(cgm.to(device), tsm.to(device), img_feat.to(device), has_meal.to(device))
    return pred, target.to(device)


def h_entry(d, h):
    if str(h) in d:
        return d[str(h)]
    if h in d:
        return d[h]
    raise KeyError(f'Horizon {h} not found. Keys={list(d.keys())}')


def p_entry(d, pid):
    if str(pid) in d:
        return d[str(pid)]
    if pid in d:
        return d[pid]
    raise KeyError(f'Participant {pid} not found. Keys={list(d.keys())}')


def result_path(config_id, seed):
    return SEED_RESULT_DIR / f'{config_id}__seed{seed}.json'

In [9]:
baseline_by_seed = {}

for seed in SEEDS:
    path = BASELINE_RESULTS_DIR / f'gated_cgm_time_seed{seed}_results.json'
    if not path.exists():
        raise FileNotFoundError(f'Missing baseline result: {path}')

    with open(path) as f:
        r = json.load(f)

    baseline_by_seed[seed] = {
        int(h): float(h_entry(r['val_results'], h)['rmse'])
        for h in HORIZONS
    }

baseline_df = pd.DataFrame.from_dict(baseline_by_seed, orient='index')
baseline_df.index.name = 'seed'
baseline_df.columns = [f'rmse_{h}' for h in baseline_df.columns]

display(baseline_df.round(4))

,rmse_15,rmse_30,rmse_60,rmse_90,rmse_120
seed,,,,,
7,10.5760,16.6694,24.0796,28.4527,31.2947
14,10.6479,16.7263,24.1044,28.3922,31.2335
21,10.5938,16.6580,24.0522,28.3216,31.1125
28,10.4778,16.5493,23.9988,28.4047,31.2741
35,10.4813,16.6073,24.1134,28.4409,31.2625
42,10.5140,16.6368,23.9722,28.3538,31.2148
49,10.6880,16.7477,24.0997,28.3797,31.1949
56,10.6339,16.7684,24.1647,28.4863,31.2816
63,10.5760,16.7480,24.0524,28.3896,31.2817


In [10]:
CHECK_SEED = SEEDS[0]
CHECK_DROPOUT = float(anchor['img_dropout'])

base_model = load_frozen_baseline(
    CHECK_SEED,
    BASELINE_CHECKPOINT_DIR,
)

train.set_seed(CHECK_SEED)

check_model = FrozenImageOnlyResidualPredictor(
    base_model=base_model,
    img_feat_dim=IMG_FEAT_DIM,
    img_proj_dim=EXPECTED_IMG_PROJ_DIM,
    img_dropout=CHECK_DROPOUT,
)

baseline_trainable = [
    name for name, p in check_model.named_parameters()
    if p.requires_grad and (
        name.startswith('gru.')
        or name.startswith('time_proj.')
        or name.startswith('base_head.')
    )
]
if baseline_trainable:
    raise RuntimeError(f'Frozen baseline contains trainable parameters: {baseline_trainable}')

trainable_names = [name for name, p in check_model.named_parameters() if p.requires_grad]
print('Trainable parameters:')
for name in trainable_names:
    print(' ', name)

batch = next(iter(val_loader))
cgm, tsm, has_meal, img_feat, _ = batch

check_model.eval()
base_model.eval()
with torch.no_grad():
    base_pred = base_model(cgm, tsm, has_meal)
    residual_pred = check_model(cgm, tsm, img_feat, has_meal)

init_diff = (base_pred - residual_pred).abs().max().item()
print(f'Initial max |residual - baseline| = {init_diff:.8f}')

if init_diff > 1e-5:
    raise RuntimeError(f'Residual initialisation sanity check failed: {init_diff}')

print('Sanity checks passed.')

Trainable parameters:
  img_proj.0.weight
  img_proj.0.bias
  image_head.0.weight
  image_head.0.bias
  image_head.3.weight
  image_head.3.bias
Initial max |residual - baseline| = 0.00000000
Sanity checks passed.


In [12]:
for config_idx, row in configs.iterrows():

    config_id = str(row['config_id'])
    img_proj_dim = int(row['img_proj_dim'])
    img_dropout = float(row['img_dropout'])
    lr = float(row['lr'])
    weight_decay = float(row['weight_decay'])

    print('\n' + '=' * 90)
    print(
        f'{config_id} ({config_idx + 1}/{len(configs)}) | '
        f'd={img_proj_dim} | dropout={img_dropout:g} | '
        f'lr={lr:g} | wd={weight_decay:g}'
    )
    print('=' * 90)

    for seed in SEEDS:

        save_path = result_path(config_id, seed)

        if save_path.exists():
            print(f'  seed={seed}: already complete')
            continue

        print(f'  seed={seed}: training')

        base_model = load_frozen_baseline(seed, BASELINE_CHECKPOINT_DIR)

        train.set_seed(seed)

        model = FrozenImageOnlyResidualPredictor(
            base_model=base_model, img_feat_dim=IMG_FEAT_DIM, img_proj_dim=img_proj_dim, img_dropout=img_dropout)

        baseline_trainable = [name for name, p in model.named_parameters()
                              if p.requires_grad and name.startswith(('gru.', 'time_proj.', 'base_head.'))]
        if baseline_trainable:
            raise RuntimeError(
                f'{config_id}, seed={seed}: baseline not fully frozen: '
                f'{baseline_trainable}'
            )

        # Verify zero-residual initialisation for every run.
        batch = next(iter(val_loader))
        cgm, tsm, has_meal, img_feat, _ = batch

        model.eval()
        base_model.eval()
        with torch.no_grad():
            base_pred = base_model(cgm, tsm, has_meal)
            residual_pred = model(cgm, tsm, img_feat, has_meal)

        init_diff = (base_pred - residual_pred).abs().max().item()
        if init_diff > 1e-5:
            raise RuntimeError(
                f'{config_id}, seed={seed}: initial residual != baseline '
                f'(max diff={init_diff})'
            )

        checkpoint_dir = CHECKPOINT_ROOT / config_id / f'seed_{seed}'

        t0 = time.time()

        model, history = train.train_model(
            model, train_loader, val_loader, stats['cgm_mean'], stats['cgm_std'], lr=lr, weight_decay=weight_decay,
            forward_fn=forward_fn, checkpoint_dir=checkpoint_dir, verbose=False, progress_bar=True, progress_desc=f'CLIP {config_id} seed={seed}')

        elapsed_seconds = time.time() - t0

        best_epoch = int(np.argmin(history['val_loss']) + 1)
        n_epochs_trained = int(len(history['train_loss']))

        y_pred_val, y_true_val = train.predict(model, val_loader, forward_fn=forward_fn)

        overall_val = metrics.evaluate(y_pred_val, y_true_val, stats['cgm_mean'], stats['cgm_std'])

        per_participant_val = metrics.evaluate_per_participant(
            y_pred_val, y_true_val, val_participant_ids, stats['cgm_mean'], stats['cgm_std'])

        # Ensure all expected validation participants were produced.
        for pid in VAL_PARTICIPANT_IDS:
            _ = p_entry(per_participant_val, pid)

        payload = {
            'encoder': ENCODER_NAME,
            'config_id': config_id,
            'config_order': int(row['config_order']),
            'is_anchor': bool(str(row['is_anchor']).lower() in ['true', '1']),
            'shared_config_sha256': config_sha256,
            'seed': int(seed),
            'hyperparameters': {
                'img_proj_dim': img_proj_dim,
                'img_dropout': img_dropout,
                'lr': lr,
                'weight_decay': weight_decay,
            },
            'initial_max_abs_diff_vs_baseline': float(init_diff),
            'best_epoch': best_epoch,
            'n_epochs_trained': n_epochs_trained,
            'elapsed_seconds': float(elapsed_seconds),
            'val_results': overall_val,
            'val_results_per_participant': per_participant_val,
        }

        with open(save_path, 'w') as f:
            json.dump(payload, f, indent=2)

        print(
            f'    saved | best_epoch={best_epoch} | '
            f'{elapsed_seconds/60:.1f} min'
        )

print('\nAll requested CLIP tuning runs are complete.')


cfg_01 (1/40) | d=4 | dropout=0.3 | lr=0.0001 | wd=1e-05
  seed=7: already complete
  seed=14: already complete
  seed=21: already complete
  seed=28: already complete
  seed=35: already complete
  seed=42: already complete
  seed=49: already complete
  seed=56: already complete
  seed=63: already complete
  seed=70: already complete

cfg_02 (2/40) | d=4 | dropout=0.1 | lr=3e-05 | wd=0
  seed=7: already complete
  seed=14: already complete
  seed=21: already complete
  seed=28: already complete
  seed=35: training
device: cuda (NVIDIA RTX PRO 6000 Blackwell Server Edition)  model: FrozenImageOnlyResidualPredictor  params: 1,029,502


CLIP cfg_02 seed=35:   0%|          | 0/50 [00:00<?, ?it/s]

    saved | best_epoch=7 | 0.3 min
  seed=42: training
device: cuda (NVIDIA RTX PRO 6000 Blackwell Server Edition)  model: FrozenImageOnlyResidualPredictor  params: 1,029,502


CLIP cfg_02 seed=42:   0%|          | 0/50 [00:00<?, ?it/s]

    saved | best_epoch=10 | 0.4 min
  seed=49: training
device: cuda (NVIDIA RTX PRO 6000 Blackwell Server Edition)  model: FrozenImageOnlyResidualPredictor  params: 1,029,502


...

    saved | best_epoch=27 | 0.8 min
  seed=63: training
device: cuda (NVIDIA RTX PRO 6000 Blackwell Server Edition)  model: FrozenImageOnlyResidualPredictor  params: 1,029,502


CLIP cfg_40 seed=63:   0%|          | 0/50 [00:00<?, ?it/s]

    saved | best_epoch=30 | 0.8 min
  seed=70: training
device: cuda (NVIDIA RTX PRO 6000 Blackwell Server Edition)  model: FrozenImageOnlyResidualPredictor  params: 1,029,502


CLIP cfg_40 seed=70:   0%|          | 0/50 [00:00<?, ?it/s]

    saved | best_epoch=28 | 0.8 min

All requested CLIP tuning runs are complete.


In [14]:
trial_rows = []
participant_rows_all = []

for _, cfg in configs.iterrows():

    config_id = str(cfg['config_id'])
    seed_results = []

    for seed in SEEDS:
        with open(result_path(config_id, seed)) as f:
            r = json.load(f)

        if r.get('shared_config_sha256') != config_sha256:
            raise RuntimeError(
                f'{config_id}, seed={seed} was produced from a different '
                'shared configuration file.'
            )

        seed_results.append(r)

    row = {
        'config_order': int(cfg['config_order']),
        'config_id': config_id,
        'is_anchor': bool(str(cfg['is_anchor']).lower() in ['true', '1']),
        'img_proj_dim': int(cfg['img_proj_dim']),
        'img_dropout': float(cfg['img_dropout']),
        'lr': float(cfg['lr']),
        'weight_decay': float(cfg['weight_decay']),
    }

    all_relative_improvements = []
    improved_horizon_seed_pairs = 0
    all_absolute_rmses = []

    for h in HORIZONS:

        candidate = np.asarray([float(h_entry(r['val_results'], h)['rmse']) for r in seed_results])

        baseline = np.asarray([baseline_by_seed[seed][int(h)] for seed in SEEDS])

        improvement_pct = (baseline - candidate) / baseline * 100.0

        # Participant means across seeds, then SD across participants.
        participant_means = []

        for pid in VAL_PARTICIPANT_IDS:
            pid_seed_rmses = [
                float(h_entry(p_entry(r['val_results_per_participant'], pid), h)['rmse']) for r in seed_results]

            pid_mean = float(np.mean(pid_seed_rmses))
            participant_means.append(pid_mean)

            participant_rows_all.append({
                'config_id': config_id,
                'participant_id': pid,
                'horizon': int(h),
                'mean_rmse_across_seeds': pid_mean,
                'sd_rmse_across_seeds': float(np.std(pid_seed_rmses, ddof=1)),
            })

        rmse_mean = float(np.mean(candidate))
        sd1 = float(np.std(candidate, ddof=1))
        sd2 = float(np.std(participant_means, ddof=1))
        wins = int(np.sum(candidate < baseline))

        row[f'rmse_{h}'] = rmse_mean
        row[f'sd1_{h}'] = sd1
        row[f'sd2_{h}'] = sd2
        row[f'improvement_pct_{h}'] = float(np.mean(improvement_pct))
        row[f'improved_seeds_{h}'] = wins

        all_relative_improvements.extend(improvement_pct.tolist())
        improved_horizon_seed_pairs += wins
        all_absolute_rmses.extend(candidate.tolist())

    row['mean_relative_improvement_pct'] = float(
        np.mean(all_relative_improvements)
    )
    row['improved_horizon_seed_pairs'] = int(improved_horizon_seed_pairs)
    row['mean_validation_rmse'] = float(np.mean(all_absolute_rmses))

    trial_rows.append(row)

trial_df = pd.DataFrame(trial_rows)
participant_df = pd.DataFrame(participant_rows_all)

trial_df.to_csv(TRIAL_TABLE_PATH, index=False)

print('Saved complete tuning table:')
print(TRIAL_TABLE_PATH)

display(
    trial_df[
        [
            'config_order','config_id','is_anchor',
            'img_dropout','lr','weight_decay',
            'mean_relative_improvement_pct',
            'improved_horizon_seed_pairs',
            'mean_validation_rmse',
            *[f'rmse_{h}' for h in HORIZONS],
        ]
    ].round(5)
)

Saved complete tuning table:
/content/drive/MyDrive/dissertation/experiments/5_residual_hyper/results/clip/summary/clip_tuning_trials.csv


,config_order,config_id,is_anchor,img_dropout,lr,weight_decay,mean_relative_improvement_pct,improved_horizon_seed_pairs,mean_validation_rmse,rmse_15,rmse_30,rmse_60,rmse_90,rmse_120
0,1,cfg_01,True,0.3,0.00010,0.00001,0.57555,50,22.05098,10.54904,16.61147,23.91076,28.19468,30.98894
1,2,cfg_02,False,0.1,0.00003,0.00000,0.67048,50,22.02707,10.54481,16.60033,23.88595,28.16165,30.94262
2,3,cfg_03,False,0.5,0.00003,0.00001,0.48249,50,22.07457,10.55275,16.62245,23.93557,28.22677,31.03529
3,4,cfg_04,False,0.1,0.00001,0.00000,0.67344,50,22.02621,10.54492,16.60057,23.88440,28.15916,30.94198
4,5,cfg_05,False,0.3,0.00100,0.00000,0.22636,45,22.14108,10.55662,16.64490,24.02936,28.34902,31.12548
5,6,cfg_06,False,0.5,0.00100,0.00000,0.38849,50,22.09668,10.55721,16.63817,23.97117,28.25904,31.05780
6,7,cfg_07,False,0.5,0.00010,0.00000,0.43470,50,22.08681,10.55391,16.62779,23.95052,28.24716,31.05466
7,8,cfg_08,False,0.1,0.00100,0.00000,-0.05636,24,22.21125,10.56569,16.68109,24.12965,28.46831,31.21150
8,9,cfg_09,False,0.0,0.00010,0.00010,0.70851,50,22.01680,10.54454,16.59767,23.87413,28.14439,30.92327
9,10,cfg_10,False,0.5,0.00003,0.00010,0.48353,50,22.07436,10.55262,16.62211,23.93539,28.22693,31.03474


In [15]:
ranked_df = trial_df.sort_values(
    by=['mean_relative_improvement_pct', 'improved_horizon_seed_pairs', 'mean_validation_rmse', 'config_order'],
    ascending=[False, False, True, True]).reset_index(drop=True)

ranked_df.insert(0, 'rank', np.arange(1, len(ranked_df) + 1))

display(
    ranked_df[
        [
            'rank','config_id','is_anchor',
            'img_dropout','lr','weight_decay',
            'mean_relative_improvement_pct',
            'improved_horizon_seed_pairs',
            'mean_validation_rmse',
            *[f'rmse_{h}' for h in HORIZONS],
            *[f'sd1_{h}' for h in HORIZONS],
            *[f'sd2_{h}' for h in HORIZONS],
        ]
    ].round(5)
)

best = ranked_df.iloc[0]
best_config_id = str(best['config_id'])

best_payload = {
    'encoder': ENCODER_NAME,
    'shared_config_sha256': config_sha256,
    'selection_data': 'validation only',
    'selection_rule': {
        'primary': (
            'maximise mean relative RMSE improvement versus seed-matched '
            'CGM+Time across all five horizons and ten seeds'
        ),
        'tie_breaker_1': 'maximise improved horizon-seed pairs',
        'tie_breaker_2': 'minimise mean validation RMSE',
        'tie_breaker_3': 'lower config_order',
    },
    'selected': {
        'config_id': best_config_id,
        'config_order': int(best['config_order']),
        'img_proj_dim': int(best['img_proj_dim']),
        'img_dropout': float(best['img_dropout']),
        'lr': float(best['lr']),
        'weight_decay': float(best['weight_decay']),
        'mean_relative_improvement_pct': float(best['mean_relative_improvement_pct']),
        'improved_horizon_seed_pairs': int(best['improved_horizon_seed_pairs']),
        'mean_validation_rmse': float(best['mean_validation_rmse']),
    },
    'horizons': {},
}

for h in HORIZONS:
    best_payload['horizons'][str(h)] = {
        'rmse_mean': float(best[f'rmse_{h}']),
        'sd1_across_seeds': float(best[f'sd1_{h}']),
        'sd2_across_participants': float(best[f'sd2_{h}']),
        'mean_improvement_pct_vs_cgm_time': float(best[f'improvement_pct_{h}']),
        'improved_seeds_vs_cgm_time': int(best[f'improved_seeds_{h}']),
    }

with open(BEST_CONFIG_PATH, 'w') as f:
    json.dump(best_payload, f, indent=2)

selected_participant_df = participant_df[participant_df['config_id'] == best_config_id].copy()

selected_participant_df.to_csv(PARTICIPANT_TABLE_PATH, index=False)

print('\n=== SELECTED CLIP CONFIGURATION ===')
print('config_id:', best_config_id)
print('dropout:', best['img_dropout'])
print('lr:', best['lr'])
print('weight_decay:', best['weight_decay'])
print(
    'mean relative improvement:',
    f"{best['mean_relative_improvement_pct']:.4f}%"
)
print(
    'improved horizon-seed pairs:',
    f"{int(best['improved_horizon_seed_pairs'])}/{len(HORIZONS)*len(SEEDS)}"
)

print('\nPer-horizon validation result: mean ± sd1 (sd2)')
for h in HORIZONS:
    print(
        f'{h:>3} min: '
        f"{best[f'rmse_{h}']:.3f} ± "
        f"{best[f'sd1_{h}']:.3f} "
        f"({best[f'sd2_{h}']:.3f}) | "
        f"improvement={best[f'improvement_pct_{h}']:.3f}% | "
        f"better seeds={int(best[f'improved_seeds_{h}'])}/{len(SEEDS)}"
    )

print('\nSaved:')
print(' ', TRIAL_TABLE_PATH)
print(' ', BEST_CONFIG_PATH)
print(' ', PARTICIPANT_TABLE_PATH)

,rank,config_id,is_anchor,img_dropout,lr,weight_decay,mean_relative_improvement_pct,improved_horizon_seed_pairs,mean_validation_rmse,rmse_15,...,sd1_15,sd1_30,sd1_60,sd1_90,sd1_120,sd2_15,sd2_30,sd2_60,sd2_90,sd2_120
0,1,cfg_09,False,0.0,0.00010,0.00010,0.70851,50,22.01680,10.54454,...,0.05993,0.07260,0.07161,0.07282,0.08149,2.07985,3.61591,7.03353,10.21955,12.04793
1,2,cfg_32,False,0.0,0.00001,0.00000,0.70540,50,22.01819,10.54337,...,0.05985,0.07048,0.07413,0.08397,0.09426,2.08039,3.61866,7.04013,10.22522,12.05574
2,3,cfg_36,False,0.0,0.00003,0.00000,0.70161,50,22.01892,10.54414,...,0.06018,0.07204,0.07582,0.07908,0.09496,2.08042,3.61852,7.04022,10.22642,12.05841
3,4,cfg_17,False,0.0,0.00003,0.00000,0.70128,50,22.01901,10.54409,...,0.06015,0.07188,0.07493,0.07848,0.09514,2.08035,3.61863,7.04107,10.22724,12.05901
4,5,cfg_23,False,0.0,0.00010,0.00000,0.69631,50,22.01978,10.54487,...,0.05973,0.07384,0.07920,0.08282,0.09074,2.08018,3.61762,7.03987,10.22619,12.05442
5,6,cfg_40,False,0.1,0.00001,0.00000,0.67596,50,22.02560,10.54488,...,0.06034,0.07091,0.07500,0.08197,0.09328,2.08177,3.61676,7.03900,10.22911,12.06355
6,7,cfg_33,False,0.1,0.00001,0.00001,0.67526,50,22.02575,10.54495,...,0.06056,0.07124,0.07663,0.08569,0.09541,2.08131,3.61626,7.03831,10.22805,12.06232
7,8,cfg_04,False,0.1,0.00001,0.00000,0.67344,50,22.02621,10.54492,...,0.06041,0.07111,0.07589,0.08379,0.09397,2.08162,3.61662,7.03986,10.22990,12.06405
8,9,cfg_37,False,0.1,0.00003,0.00001,0.67333,50,22.02633,10.54486,...,0.06044,0.07170,0.07637,0.08286,0.09583,2.08159,3.61806,7.03949,10.22784,12.06045
9,10,cfg_02,False,0.1,0.00003,0.00000,0.67048,50,22.02707,10.54481,...,0.06049,0.07143,0.07302,0.07935,0.09340,2.08215,3.61862,7.04208,10.23071,12.06284



=== SELECTED CLIP CONFIGURATION ===
config_id: cfg_09
dropout: 0.0
lr: 0.0001
weight_decay: 0.0001
mean relative improvement: 0.7085%
improved horizon-seed pairs: 50/50

Per-horizon validation result: mean ± sd1 (sd2)
 15 min: 10.545 ± 0.060 (2.080) | improvement=0.297% | better seeds=10/10
 30 min: 16.598 ± 0.073 (3.616) | improvement=0.496% | better seeds=10/10
 60 min: 23.874 ± 0.072 (7.034) | improvement=0.811% | better seeds=10/10
 90 min: 28.144 ± 0.073 (10.220) | improvement=0.905% | better seeds=10/10
120 min: 30.923 ± 0.081 (12.048) | improvement=1.033% | better seeds=10/10

Saved:
  /content/drive/MyDrive/dissertation/experiments/5_residual_hyper/results/clip/summary/clip_tuning_trials.csv
  /content/drive/MyDrive/dissertation/experiments/5_residual_hyper/results/clip/summary/clip_best_config.json
  /content/drive/MyDrive/dissertation/experiments/5_residual_hyper/results/clip/summary/clip_selected_config_participant_rmse.csv
